# **[트랙 2-1] ROS2 하드웨어 인터페이스 및 모터 제어 실무**<br><br>
---
---
<br>

## 0. 비상용

```bash
pkill -9 gz
pkill -9 gz_sim
pkill -9 ruby
```

---

## 1. ROS2 Jazzy 설치

```bash
sudo apt install software-properties-common
sudo add-apt-repository universe

sudo apt update
sudo apt install curl -y

export ROS_APT_SOURCE_VERSION=$(curl -s https://api.github.com/repos/ros-infrastructure/ros-apt-source/releases/latest | grep -F "tag_name" | awk -F'"' '{print $4}')

curl -L -o /tmp/ros2-apt-source.deb "https://github.com/ros-infrastructure/ros-apt-source/releases/download/${ROS_APT_SOURCE_VERSION}/ros2-apt-source_${ROS_APT_SOURCE_VERSION}.$(. /etc/os-release && echo ${UBUNTU_CODENAME:-${VERSION_CODENAME}})_all.deb"

sudo dpkg -i /tmp/ros2-apt-source.deb

sudo apt update
sudo apt upgrade

sudo apt install ros-dev-tools
sudo apt install ros-jazzy-desktop
```

---

## 2. 패키지 설치

```bash
sudo apt update
sudo apt install ros-jazzy-tf2-ros \
                 ros-jazzy-tf-transformations \
                 ros-jazzy-rmw-cyclonedds-cpp \
                 ros-jazzy-turtlebot3-gazebo \
                 ros-jazzy-turtlebot3-description \
                 ros-jazzy-turtlebot3-teleop \
                 ros-jazzy-ros-gz \
                 ros-jazzy-ros-gz-sim \
                 ros-jazzy-ros-gz-bridge \
                 ros-jazzy-rviz2 \
                 ros-jazzy-slam-toolbox \
                 ros-jazzy-nav2-map-server \
                 ros-jazzy-navigation2 ros-jazzy-nav2-bringup
```

---

## 3. ROS2 개발 환경 설정

```bash
echo "source /opt/ros/jazzy/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 4. ROS2 통신 환경 설정

```bash
echo "export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp" >> ~/.bashrc
echo "export ROS_DOMAIN_ID=0" >> ~/.bashrc
source ~/.bashrc
```

---

## 5. 작업 디렉토리 생성

```bash
mkdir -p ~/robot_navigation/{colcon_ws,firmware,scripts}
mkdir ~/robot_navigation/colcon_ws/src
```

---

## 6. colcon 환경 설정

```bash
cd ~/robot_navigation/colcon_ws
colcon build
echo "source ~/robot_navigation/colcon_ws/install/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

---

## 7. colcon 패키지 생성

**실행 디렉토리:** `~/robot_navigation/colcon_ws/src`

```bash
ros2 pkg create pkg_name --build-type ament_python --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 8. colcon 패키지 빌드

**실행 디렉토리:** `~/robot_navigation/colcon_ws`
```bash
colcon build
```
```bash
colcon build --packages-select pkg_name
```

---

## 9. 빌드 후 Source

**실행 디렉토리:** `~/robot_navigation/colcon_ws`
```bash
source install/setup.bash
```

**혹은 아무 디렉토리에서:**
```bash
source ~/robot_navigation/colcon_ws/install/setup.bash
```

---

## 10. Demo Launch 파일

```python
from launch import LaunchDescription
from launch.actions import ExecuteProcess, RegisterEventHandler
from launch.event_handlers import OnProcessStart


def generate_launch_description():

    publisher = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    processor = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    subscriber = ExecuteProcess(
        cmd=[
            'gnome-terminal', '--', 'bash', '-c',
            'source ~/robot_navigation/colcon_ws/install/setup.bash; '
            'ros2 run pkg_name py_name; '
            'exec bash'
        ],
        output='screen'
    )

    return LaunchDescription([
        publisher,

        RegisterEventHandler(
            OnProcessStart(
                target_action=publisher,
                on_start=[processor]
            )
        ),

        RegisterEventHandler(
            OnProcessStart(
                target_action=processor,
                on_start=[subscriber]
            )
        ),
    ])
```

---

## 11. 커스텀 메세지/서비스

**실행 디렉토리:** `~/robot_navigation/colcon_ws/src`
```bash
ros2 pkg create pkg_name --build-type ament_cmake --dependencies rclpy std_msgs geometry_msgs sensor_msgs nav_msgs
```

---

## 12. CMakeLists.txt (전체 복사)

```cmake
cmake_minimum_required(VERSION 3.8)
project(PKG_NAME)

find_package(ament_cmake REQUIRED)
find_package(rosidl_default_generators REQUIRED)

rosidl_generate_interfaces(${PROJECT_NAME}
  "msg/MSG_NAME.msg"
  "srv/SRV_NAME.srv"
)

ament_package()
```

---

## 13. interfaces/package.xml (코드 추가)

```xml
  <build_depend>rosidl_default_generators</build_depend>
  <exec_depend>rosidl_default_runtime</exec_depend>
  <member_of_group>rosidl_interface_packages</member_of_group>
```

---

## 14. my_pkg/package.xml (코드 추가)

```xml
  <depend>interfaces_pkg_name</depend>
```

---

## 15. Arduino IDE 설치

```bash
sudo apt update
sudo apt install -y curl libfuse2t64
```

```bash
sudo mkdir -p /opt/arduino-ide
cd /opt/arduino-ide
```

```bash
sudo curl -fL \
    https://github.com/arduino/arduino-ide/releases/latest/download/arduino-ide_2.3.10_Linux_64bit.AppImage \
    -o arduino-ide.AppImage
```

---

## 16. Arduino IDE 테스트 실행

```bash
cd /opt/arduino-ide
sudo chmod +x arduino-ide.AppImage
./arduino-ide.AppImage --no-sandbox
```

---

## 17. Arduino IDE 간편 커스텀 명령어 생성

```bash
nano ~/robot_navigation/scripts/arduino-ide
```

```bash
#!/bin/sh
exec /opt/arduino-ide/arduino-ide.AppImage --no-sandbox "$@"
```

```bash
chmod +x ~/robot_navigation/scripts/arduino-ide
sudo ln -s $HOME/robot_navigation/scripts/arduino-ide /usr/local/bin/arduino-ide
```

---

## 18. Arduino IDE 커스텀 명령어 테스트

```bash
arduino-ide
```

---

## 19. Serial Port 권한 확인

```bash
groups
```

---

## 20. Serial Port 권한 부여

```bash
sudo usermod -a -G dialout $USER
```

이후

```bash
reboot
```

<br>

#### 재부팅 후에 Serial Port 권한 재확인:
```bash
groups
ls -l /dev/ttyUSB*
```

---

## 21. ESP-IDF 설치

#### EIM (ESP-IDF Installation Manager) 설치:
```bash
echo "deb [trusted=yes] https://dl.espressif.com/dl/eim/apt/ stable main" | sudo tee /etc/apt/sources.list.d/espressif.list
sudo apt update
sudo apt install eim
```

#### EIM 실행:
```bash
eim
```

---

## 22. ESP-IDF 독립 실행 환경 구성

#### ESP-IDF 환경 활성화 스크립트 경로 확인 (경로 복사):
```bash
find ~/.espressif -type f -name 'activate_idf_v5.5.5.sh' 2>/dev/null
```

<br>

#### ESP-IDF 전용 Shell 스크립트 생성:
```bash
nano ~/robot_navigation/scripts/idfsh
```

<br>

#### 스크립트 작성:
```bash
#!/usr/bin/env bash

env -i \
HOME="$HOME" \
USER="$USER" \
TERM="$TERM" \
DISPLAY="${DISPLAY:-}" \
XAUTHORITY="${XAUTHORITY:-}" \
PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin" \
bash --noprofile --norc -c '
    source "$HOME/<복사한_ACTIVATE_IDF_SH_PATH>"
    export PATH="$IDF_PATH/tools:$PATH"
    export PATH="$HOME/.local/bin:$PATH"
    exec bash --noprofile --norc
'
```

<br>

#### 스크립트 실행 권한 부여:
```bash
chmod +x ~/robot_navigation/scripts/idfsh
```

<br>

#### 실행 명령어 등록:
```bash
sudo ln -sf /usr/bin/as /usr/local/bin/as
sudo ln -sf /usr/bin/ld /usr/local/bin/ld
sudo ln -s ~/robot_navigation/scripts/idfsh /usr/local/bin/idfsh
```

<br>

#### 명령어 테스트:
```bash
echo $ROS_DISTRO
idfsh
echo $ROS_DISTRO
idf.py --version
```

---

## 23. ESP-IDF 기본 예제 프로젝트 생성 및 빌드 검증

#### ESP-IDF 기본 예제 복사:
```bash
cp -a "$IDF_PATH/examples/get-started/hello_world/." ~/robot_navigation/firmware/esp32/
```

<br>

#### 컴파일 대상 칩을 ESP32로 지정 및 빌드:
```bash
cd ~/robot_navigation/firmware/esp32
idf.py set-target esp32
idf.py build
```

<br>

#### ESP32에 펌웨어 Flash:
```bash
idfsh
cd ~/robot_navigation/firmware/esp32/
idf.py -p /dev/ttyUSB0 flash
```

- 실패하면 "BOOT 버튼 누른 상태로 flash하고, Connecting... 나오면 놓기"

<br>

#### Serial Monitor:
```bash
idf.py -p /dev/ttyUSB0 monitor
```

- flash와 monitor 동시에 하려면 `idf.py -p /dev/ttyUSB0 flash monitor`

---

## 24. micro-ROS 의존성 설치

```bash
idfsh
python -m pip install catkin_pkg colcon-common-extensions lark
```

---

## 25. 기존 파일 백업

```bash
mv main/hello_world_main.c main/hello_world_main.c.backup
cp main/CMakeLists.txt main/CMakeLists.txt.backup
```

#### 체크:
```text
ls main
>> CMakeLists.txt  CMakeLists.txt.backup  hello_world_main.c.backup
```

---

## 26. micro-ROS dependency 파일 만들기
```bash
cd ~/robot_navigation/firmware/esp32/
nano main/idf_component.yml
```

#### 작성:
```yml
dependencies:
  micro-ros/micro_ros_espidf_component:
    version: "^24.0.0"
```

<br>

```bash
nano app-colcon.meta
```

#### 작성:
```json
{
    "names": {
        "rmw_microxrcedds": {
            "cmake-args": [
                "-DRMW_UXRCE_TRANSPORT=custom"
            ]
        }
    }
}
```

---

## 27. Serial Transport Header 만들기

```bash
nano main/esp32_serial_transport.h
```

#### 작성:
```c
#ifndef ESP32_SERIAL_TRANSPORT_H
#define ESP32_SERIAL_TRANSPORT_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#include <uxr/client/transport.h>

#ifdef __cplusplus
extern "C" {
#endif

bool esp32_serial_open(
    struct uxrCustomTransport *transport);

bool esp32_serial_close(
    struct uxrCustomTransport *transport);

size_t esp32_serial_write(
    struct uxrCustomTransport *transport,
    const uint8_t *buffer,
    size_t length,
    uint8_t *error);

size_t esp32_serial_read(
    struct uxrCustomTransport *transport,
    uint8_t *buffer,
    size_t length,
    int timeout,
    uint8_t *error);

#ifdef __cplusplus
}
#endif

#endif
```

---

## 28. Serial Transport Header 만들기

```bash
nano main/esp32_serial_transport.c
```

#### 작성:
```c
#include "esp32_serial_transport.h"

#include "driver/uart.h"
#include "driver/gpio.h"

#include "sdkconfig.h"

#define UART_BUFFER_SIZE 512

#define UART_TXD CONFIG_MICROROS_UART_TXD
#define UART_RXD CONFIG_MICROROS_UART_RXD
#define UART_RTS CONFIG_MICROROS_UART_RTS
#define UART_CTS CONFIG_MICROROS_UART_CTS


bool esp32_serial_open(
    struct uxrCustomTransport *transport)
{
    size_t *uart_port = (size_t *)transport->args;

    uart_config_t uart_config = {
        .baud_rate = 115200,
        .data_bits = UART_DATA_8_BITS,
        .parity = UART_PARITY_DISABLE,
        .stop_bits = UART_STOP_BITS_1,
        .flow_ctrl = UART_HW_FLOWCTRL_DISABLE,
        .source_clk = UART_SCLK_DEFAULT,
    };

    if (uart_param_config(*uart_port, &uart_config) != ESP_OK) {
        return false;
    }

    if (uart_set_pin(
            *uart_port,
            UART_TXD,
            UART_RXD,
            UART_RTS,
            UART_CTS) != ESP_OK)
    {
        return false;
    }

    if (uart_driver_install(
            *uart_port,
            UART_BUFFER_SIZE * 2,
            0,
            0,
            NULL,
            0) != ESP_OK)
    {
        return false;
    }

    return true;
}


bool esp32_serial_close(
    struct uxrCustomTransport *transport)
{
    size_t *uart_port = (size_t *)transport->args;

    return uart_driver_delete(*uart_port) == ESP_OK;
}


size_t esp32_serial_write(
    struct uxrCustomTransport *transport,
    const uint8_t *buffer,
    size_t length,
    uint8_t *error)
{
    (void)error;

    size_t *uart_port = (size_t *)transport->args;

    int written = uart_write_bytes(
        *uart_port,
        (const char *)buffer,
        length);

    return written > 0 ? (size_t)written : 0;
}


size_t esp32_serial_read(
    struct uxrCustomTransport *transport,
    uint8_t *buffer,
    size_t length,
    int timeout,
    uint8_t *error)
{
    (void)error;

    size_t *uart_port = (size_t *)transport->args;

    int received = uart_read_bytes(
        *uart_port,
        buffer,
        length,
        pdMS_TO_TICKS(timeout));

    return received > 0 ? (size_t)received : 0;
}
```

---

## 29. main.cpp 작성 (UART 방식)

```bash
nano main/main.cpp
```

#### 작성:
```cpp
#include <unistd.h>

#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

#include "driver/uart.h"

#include <rcl/rcl.h>
#include <rclc/rclc.h>
#include <rclc/executor.h>

#include <std_msgs/msg/int32.h>

#include <rmw_microxrcedds_c/config.h>
#include <rmw_microros/rmw_microros.h>

#include "esp32_serial_transport.h"


namespace
{

rcl_publisher_t publisher;
std_msgs__msg__Int32 message;


/**
 * @brief rcl/rclc 함수의 반환값을 검사하는 치명적 오류 처리 매크로.
 *
 * 전달된 함수 fn을 실행하고 반환값이 RCL_RET_OK가 아니면
 * 현재 실행 중인 FreeRTOS task를 종료한다.
 *
 * 주로 node, timer, executor 등의 초기화처럼
 * 실패했을 때 이후 실행을 계속하기 어려운 함수에 사용한다.
 *
 * @param fn rcl_ret_t 타입의 반환값을 가지는 rcl/rclc 함수 호출식
 */
#define RCCHECK(fn)                           \
    do {                                      \
        rcl_ret_t rc = (fn);                  \
        if (rc != RCL_RET_OK) {               \
            vTaskDelete(nullptr);             \
        }                                     \
    } while (0)


/**
 * @brief rcl/rclc 함수의 반환값을 무시하는 비치명적 오류 처리 매크로.
 *
 * 전달된 함수 fn을 실행하고 반환값을 rc에 저장하지만,
 * 함수 실행이 실패하더라도 현재 task를 종료하지 않고 계속 실행한다.
 *
 * 주로 일시적으로 실패할 수 있지만 프로그램 전체를 종료할 필요는 없는
 * publish 등의 함수에 사용한다.
 *
 * @param fn rcl_ret_t 타입의 반환값을 가지는 rcl/rclc 함수 호출식
 */
#define RCSOFTCHECK(fn)                       \
    do {                                      \
        rcl_ret_t rc = (fn);                  \
        (void)rc;                             \
    } while (0)


/**
 * @brief ROS timer 만료 시 호출되는 callback 함수.
 *
 * executor가 timer의 주기가 만료된 것을 감지하면 이 함수를 호출한다.
 * 현재 message를 publisher를 통해 publish한 뒤 message.data를 1 증가시킨다.
 *
 * publish에 실패하더라도 RCSOFTCHECK를 사용하므로
 * 현재 FreeRTOS task를 종료하지 않고 계속 실행한다.
 *
 * @param timer          callback을 발생시킨 ROS timer 객체의 포인터
 * @param last_call_time 이전 callback 호출과 관련된 시간 정보(ns).
 */
void timer_callback(
    rcl_timer_t *timer,
    int64_t last_call_time)
{
    (void)last_call_time;

    if (timer == nullptr) {
        return;
    }

    RCSOFTCHECK(
        rcl_publish(
            &publisher,   // message를 전송할 publisher
            &message,     // 전송할 ROS message
            nullptr)      // 별도의 RMW publisher allocation 정보 사용하지 않음
        );

    ++message.data;
}


/**
 * @brief micro-ROS node를 초기화하고 주기적으로 executor를 실행하는 FreeRTOS task.
 * 
 * 초기화가 끝난 뒤 무한 루프에서 executor를 반복 실행하여
 * timer callback 등의 ROS callback이 처리되도록 한다.
 *
 * @param argument FreeRTOS task 생성 시 전달되는 사용자 인자.
 */
void micro_ros_task(void *argument)
{
    (void)argument;

    // ROS node/executor가 사용할 공통 실행 환경 객체
    rclc_support_t support;

    // ROS 2 rcl 계층에서 사용할 기본 memory allocator 획득
    rcl_allocator_t allocator =
        rcl_get_default_allocator();

    // ROS 초기화 옵션 객체를 안전한 zero-initialized 상태로 생성
    rcl_init_options_t init_options =
        rcl_get_zero_initialized_init_options();

    // init_options를 실제 사용 가능한 상태로 초기화
    RCCHECK(
        rcl_init_options_init(
            &init_options,
            allocator));

    // 이 micro-ROS node가 사용할 ROS Domain ID
    constexpr size_t ros_domain_id = 본인_도메인_숫자;

    // init_options에 Domain ID 설정 추가
    RCCHECK(
        rcl_init_options_set_domain_id(
            &init_options,
            ros_domain_id));

    // support 초기화 및 공통 실행 환경 준비
    RCCHECK(
        rclc_support_init_with_options(
            &support,       // 초기화할 rclc support 객체
            0,              // argc
            nullptr,        // argv
            &init_options,  // Domain ID, RMW, Agent 설정 등이 들어 있는 ROS 초기화 옵션
            &allocator));   // ROS 객체 초기화에 사용할 memory allocator

    // ROS node 객체
    rcl_node_t node;

    // ROS node 생성
    RCCHECK(
        rclc_node_init_default(
            &node,                  // 생성할 노드
            "my_esp32_publisher",   // 노드 이름
            "",                     // 노드 namespace
            &support));


    // std_msgs.msg.Int32 메세지 타입의 publisher 생성
    rclc_publisher_init_default(
        &publisher,
        &node,
        ROSIDL_GET_MSG_TYPE_SUPPORT(
            std_msgs,
            msg,
            Int32),
        "esp32_int32");   // publish할 topic 이름


    // ROS timer 객체
    rcl_timer_t timer;

    // timer의 주기(1000ms)
    constexpr unsigned int timer_period_ms = 1000;

    // 1초 주기의 ROS timer 생성 및 timer 만료 시 timer_callback이 호출되도록 등록
    RCCHECK(
        rclc_timer_init_default(
            &timer,
            &support,
            RCL_MS_TO_NS(timer_period_ms),
            timer_callback));


    // callback을 실행할 executor 객체
    rclc_executor_t executor;

    // executor 초기화
    RCCHECK(
        rclc_executor_init(
            &executor,
            &support.context,  // executor가 사용할 ROS context
            1,                 // executor가 관리할 handle 개수: timer 1개
            &allocator));

    // 위에서 생성한 timer를 executor에 등록
    RCCHECK(
        rclc_executor_add_timer(
            &executor,
            &timer));

    // 처음 publish할 Int32 message 값
    message.data = 0;


    // 등록된 ROS entity(timer 등)의 상태를 확인하고 실행 가능한 callback이 있으면 처리
    while (true) {
        rclc_executor_spin_some(
            &executor,
            RCL_MS_TO_NS(100));  // timeout

        usleep(10000);
    }
}


// micro-ROS serial transport에서 사용할 ESP32 UART 포트 번호
size_t uart_port = UART_NUM_0;

} // namespace


extern "C" void app_main(void)
{
#if defined(RMW_UXRCE_TRANSPORT_CUSTOM)

    rmw_uros_set_custom_transport(
        true,
        static_cast<void *>(&uart_port),
        esp32_serial_open,
        esp32_serial_close,
        esp32_serial_write,
        esp32_serial_read);

#else

#error "micro-ROS custom transport is not enabled"

#endif

    // micro-ROS 로직을 FreeRTOS task로 등록
    xTaskCreate(
        micro_ros_task,                   // 실행할 함수
        "micro_ros_task",                 // task 이름
        CONFIG_MICRO_ROS_APP_STACK,       // 스택 크기 (기본: 16000 bytes)
        nullptr,                          // 함수에 넘길 인자
        CONFIG_MICRO_ROS_APP_TASK_PRIO,   // 우선순위 (기본: 5)
        nullptr);                         // task handle 저장 안 함
}
```

---

## 30. micro-ROS 기본 설정값 정의

```bash
nano main/Kconfig.projbuild
```

#### 작성:
```text
menu "micro-ROS example-app settings"

    config MICRO_ROS_APP_STACK
        int "Stack the micro-ROS app (Bytes)"
        default 16000
        help
        Stack size in Bytes of the micro-ROS app

    config MICRO_ROS_APP_TASK_PRIO
        int "Priority of the micro-ROS app"
        default 5
        help
        Priority of micro-ros task higher value means higher priority

endmenu
```

---

## 31. micro-ROS Component 가져오기

```bash
idfsh
pwd
```

```text
>> $HOME/robot_navigation/firmware/esp32
```

<br>

```bash
idf.py reconfigure
```

```bash
ls
```

```text
>> managed_components 존재 확인
```

---

## 32. micro-ROS UART 통신 설정 (menuconfig)

```bash
pwd
```

```text
>> $HOME/robot_navigation/firmware/esp32
```

<br>

```bash
idf.py menuconfig
```

#### 설정:
```text
- "micro-ROS Settings" -> "micro-ROS middleware" -> "micro-ROS over eProsima Micro XRCE-DDS" -> "micro-ROS network interface select" -> "Micro XRCE-DDS over UART"
- "UART Settings" -> 전부 "-1"인지 확인
- "S" 눌러 저장 -> "Esc" 눌러 종료
```

#### 확인:
```text
(venv) grep MICRO_ROS_ESP_XRCE_DDS_MIDDLEWARE sdkconfig
>> CONFIG_MICRO_ROS_ESP_XRCE_DDS_MIDDLEWARE=y

(venv) grep MICRO_ROS_ESP_UART_TRANSPORT sdkconfig
>> CONFIG_MICRO_ROS_ESP_UART_TRANSPORT=y

(venv) grep MICROROS_UART sdkconfig
>> CONFIG_MICROROS_UART_TXD=-1
>> CONFIG_MICROROS_UART_RXD=-1
>> CONFIG_MICROROS_UART_RTS=-1
>> CONFIG_MICROROS_UART_CTS=-1
```

---

## 33. UART transport를 사용하는 micro-ROS 펌웨어 빌드

```bash
pwd
```

```text
>> $HOME/robot_navigation/firmware/esp32
```

<br>

```bash
idf.py build
```

```text
>> Project build complete
```

---

## 34. micro-ROS 펌웨어 Flash

```bash
idf.py -p /dev/ttyUSB0 flash
```

- 여기서는 Serial Monitor 불가능 (Serial Port Busy)

---

## 35. ROS2 추가 필수 패키지 설치

#### 새 터미널 실행:

```bash
sudo apt update
sudo apt install -y \
    ros-jazzy-tf2-ros \
    ros-jazzy-tf-transformations \
    ros-jazzy-rmw-cyclonedds-cpp \
    ros-jazzy-turtlebot3-gazebo \
    ros-jazzy-turtlebot3-description \
    ros-jazzy-turtlebot3-teleop \
    ros-jazzy-ros-gz \
    ros-jazzy-ros-gz-sim \
    ros-jazzy-ros-gz-bridge \
    ros-jazzy-rviz2 \
    ros-jazzy-slam-toolbox \
    ros-jazzy-nav2-map-server \
    ros-jazzy-navigation2 \
    ros-jazzy-nav2-bringup \
    python3-rosdep \
    python3-vcstool \
    build-essential \
    libcurl4-openssl-dev
```

---

## 36. Agent 빌드

```bash
rosdep update

cd ~/robot_navigation/colcon_ws

git clone -b jazzy --depth 1 \
https://github.com/micro-ROS/micro_ros_setup.git \
src/micro_ros_setup

rosdep install \
    --from-paths src \
    --ignore-src \
    -y
    
cd ~/robot_navigation/colcon_ws
colcon build --symlink-install --packages-select micro_ros_setup
source install/setup.bash

ros2 run micro_ros_setup create_agent_ws.sh
ros2 run micro_ros_setup build_agent.sh
source install/setup.bash
```

---

## 37. Agent 실행

```bash
ros2 run micro_ros_agent micro_ros_agent \
    serial \
    --dev /dev/ttyUSB0 \
    -b 115200 \
    -v6
```

- 이후 ESP 리셋버튼

<br>

#### 새 터미널:

```bash
ros2 node list
```

```text
>> /my_esp32_publisher
```

```bash
ros2 topic list -t
```

```text
>> /esp32_int32 [std_msgs/msg/Int32]
```

```bash
ros2 topic echo /esp32_int32 
```

```text
>> data: 0
>> ---
>> data: 1
>> ---
```

---

## 38. 기존 파일 백업
```bash
cp main/main.cpp main/main.cpp.uart.backup
cp main/CMakeLists.txt main/CMakeLists.txt.uart.backup
mv app-colcon.meta app-colcon.meta.backup
```

---

## 39. IP 주소 확인
```bash
hostname -I
```

---

## 40. main.cpp 작성 (WIFI UDP 방식)
```bash
nano main/main.cpp
```

#### 작성:

```cpp
#include <unistd.h>
#include <inttypes.h>

#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

#include "esp_err.h"
#include "esp_log.h"

#include <uros_network_interfaces.h>

#include <rcl/rcl.h>
#include <rclc/rclc.h>
#include <rclc/executor.h>

#include <std_msgs/msg/int32.h>

#include <rmw_microros/rmw_microros.h>


namespace
{

rcl_publisher_t publisher;
rcl_subscription_t command_subscriber;

std_msgs__msg__Int32 publish_message;
std_msgs__msg__Int32 command_message;

constexpr char TAG[] = "micro_ros";
constexpr size_t ROS_DOMAIN_ID = 본인_도메인_숫자;


/**
 * @brief rcl/rclc 함수의 반환값을 검사하는 치명적 오류 처리 매크로.
 *
 * 전달된 함수 fn을 실행하고 반환값이 RCL_RET_OK가 아니면
 * 현재 실행 중인 FreeRTOS task를 종료한다.
 *
 * 주로 node, timer, executor 등의 초기화처럼
 * 실패했을 때 이후 실행을 계속하기 어려운 함수에 사용한다.
 *
 * @param fn rcl_ret_t 타입의 반환값을 가지는 rcl/rclc 함수 호출식
 */
#define RCCHECK(fn)                           \
    do {                                      \
        rcl_ret_t rc = (fn);                  \
        if (rc != RCL_RET_OK) {               \
            vTaskDelete(nullptr);             \
        }                                     \
    } while (0)


/**
 * @brief rcl/rclc 함수의 반환값을 무시하는 비치명적 오류 처리 매크로.
 *
 * 전달된 함수 fn을 실행하고 반환값을 rc에 저장하지만,
 * 함수 실행이 실패하더라도 현재 task를 종료하지 않고 계속 실행한다.
 *
 * 주로 일시적으로 실패할 수 있지만 프로그램 전체를 종료할 필요는 없는
 * publish 등의 함수에 사용한다.
 *
 * @param fn rcl_ret_t 타입의 반환값을 가지는 rcl/rclc 함수 호출식
 */
#define RCSOFTCHECK(fn)                       \
    do {                                      \
        rcl_ret_t rc = (fn);                  \
        (void)rc;                             \
    } while (0)


/**
 * @brief ROS timer 만료 시 호출되는 callback 함수.
 *
 * executor가 timer의 주기가 만료된 것을 감지하면 이 함수를 호출한다.
 * 현재 message를 publisher를 통해 publish한 뒤 message.data를 1 증가시킨다.
 *
 * publish에 실패하더라도 RCSOFTCHECK를 사용하므로
 * 현재 FreeRTOS task를 종료하지 않고 계속 실행한다.
 *
 * @param timer          callback을 발생시킨 ROS timer 객체의 포인터
 * @param last_call_time 이전 callback 호출과 관련된 시간 정보(ns).
 */
void timer_callback(
    rcl_timer_t *timer,
    int64_t last_call_time)
{
    (void)last_call_time;

    if (timer == nullptr) {
        return;
    }

    RCSOFTCHECK(
        rcl_publish(
            &publisher,        // message를 전송할 publisher
            &publish_message,  // 전송할 ROS message
            nullptr)           // 별도의 RMW publisher allocation 정보 사용하지 않음
        );

    ++publish_message.data;
}


/**
 * @brief subscription으로 수신한 Int32 메시지를 처리하는 callback 함수.
 *
 * executor가 subscription에서 새 메시지를 수신하면 이 함수를 호출한다.
 * 전달받은 generic pointer를 std_msgs/msg/Int32 메시지 타입의 포인터로
 * 변환한 뒤, 수신한 data 값을 로그로 출력한다.
 *
 * @param msg_in 수신한 ROS 메시지를 가리키는 포인터.
 */
void command_callback(const void *msg_in)
{
    if (msg_in == nullptr) {
        return;
    }

    const auto *received_message =
        static_cast<const std_msgs__msg__Int32 *>(msg_in);

    ESP_LOGI(
        TAG,                              // 로그 메시지에 표시할 tag
        "Received /esp32/cmd: %" PRId32,  // int32_t 값을 10진수로 출력
        received_message->data);          // 실제로 출력할 std_msgs.msg.Int32 값
}


/**
 * @brief micro-ROS node를 초기화하고 주기적으로 executor를 실행하는 FreeRTOS task.
 * 
 * 초기화가 끝난 뒤 무한 루프에서 executor를 반복 실행하여
 * timer callback 등의 ROS callback이 처리되도록 한다.
 *
 * @param argument FreeRTOS task 생성 시 전달되는 사용자 인자.
 */
void micro_ros_task(void *argument)
{
    (void)argument;

    // ROS node/executor가 사용할 공통 실행 환경 객체
    rclc_support_t support;

    // ROS 2 rcl 계층에서 사용할 기본 memory allocator 획득
    rcl_allocator_t allocator =
        rcl_get_default_allocator();

    // ROS 초기화 옵션 객체를 안전한 zero-initialized 상태로 생성
    rcl_init_options_t init_options =
        rcl_get_zero_initialized_init_options();

    // init_options를 실제 사용 가능한 상태로 초기화
    RCCHECK(
        rcl_init_options_init(
            &init_options,
            allocator));

    // init_options에 Domain ID 설정 추가
    RCCHECK(
        rcl_init_options_set_domain_id(
            &init_options,
            ROS_DOMAIN_ID));

#ifdef CONFIG_MICRO_ROS_ESP_XRCE_DDS_MIDDLEWARE

    // micro-ROS RMW 초기화 옵션
    rmw_init_options_t *rmw_options =
        rcl_init_options_get_rmw_init_options(&init_options);

    // UDP로 접속할 micro-ROS Agent 주소 설정
    RCCHECK(
        rmw_uros_options_set_udp_address(
            CONFIG_MICRO_ROS_AGENT_IP,
            CONFIG_MICRO_ROS_AGENT_PORT,
            rmw_options));

#endif

    // support 초기화 및 공통 실행 환경 준비
    RCCHECK(
        rclc_support_init_with_options(
            &support,       // 초기화할 rclc support 객체
            0,              // argc
            nullptr,        // argv
            &init_options,  // Domain ID, RMW, Agent 설정 등이 들어 있는 ROS 초기화 옵션
            &allocator));   // ROS 객체 초기화에 사용할 memory allocator

    // ROS node 객체
    rcl_node_t node;

    // ROS node 생성
    RCCHECK(
        rclc_node_init_default(
            &node,                  // 생성할 노드
            "my_esp32_publisher",   // 노드 이름
            "",                     // 노드 namespace
            &support));

    // Publisher 생성 (std_msgs.msg.Int32 메세지 타입)
    RCCHECK(
        rclc_publisher_init_default(
            &publisher,
            &node,
            ROSIDL_GET_MSG_TYPE_SUPPORT(
                std_msgs,
                msg,
                Int32),
            "esp32_int32"));   // publish할 topic 이름
    
    // Subscriber 생성
    RCCHECK(
        rclc_subscription_init_default(
            &command_subscriber,
            &node,
            ROSIDL_GET_MSG_TYPE_SUPPORT(
                std_msgs,
                msg,
                Int32),
            "esp32/cmd"));   // subscribe할 topic 이름

    // ROS timer 객체
    rcl_timer_t timer;

    // timer의 주기(1000ms)
    constexpr unsigned int timer_period_ms = 1000;

    // 1초 주기의 ROS timer 생성 및 timer 만료 시 timer_callback이 호출되도록 등록
    RCCHECK(
        rclc_timer_init_default(
            &timer,
            &support,
            RCL_MS_TO_NS(timer_period_ms),
            timer_callback));

    // callback을 실행할 executor 객체
    rclc_executor_t executor;

    // executor 초기화
    RCCHECK(
        rclc_executor_init(
            &executor,
            &support.context,  // executor가 사용할 ROS context
            2,                 // executor가 관리할 handle 개수: 2개 (timer, subscriber)
            &allocator));

    // timer를 executor에 등록
    RCCHECK(
        rclc_executor_add_timer(
            &executor,
            &timer));
    
    // subscriber를 executor에 등록
    RCCHECK(
        rclc_executor_add_subscription(
            &executor,
            &command_subscriber,
            &command_message,
            command_callback,
            ON_NEW_DATA));      // 새로운 메시지가 있을 때만 callback 실행


    // 초기 값 정의
    publish_message.data = 0;
    command_message.data = 0;


    // 로그 출력
    ESP_LOGI(TAG, "micro-ROS Node Initialized");
    ESP_LOGI(TAG, "Publishing: /esp32_int32");
    ESP_LOGI(TAG, "Subscribing: /esp32/cmd");


    // 등록된 ROS entity(timer 등)의 상태를 확인하고 실행 가능한 callback이 있으면 처리
    while (true) {
        rclc_executor_spin_some(
            &executor,
            RCL_MS_TO_NS(100));  // timeout

        usleep(10000);
    }
}

} // namespace


extern "C" void app_main(void)
{
    // Wi-Fi 네트워크 인터페이스를 초기화하고 연결
    ESP_ERROR_CHECK(
        uros_network_interface_initialize());

    // micro-ROS 로직을 FreeRTOS task로 등록
    xTaskCreate(
        micro_ros_task,                   // 실행할 함수
        "micro_ros_task",                 // task 이름
        CONFIG_MICRO_ROS_APP_STACK,       // 스택 크기 (기본: 16000 bytes)
        nullptr,                          // 함수에 넘길 인자
        CONFIG_MICRO_ROS_APP_TASK_PRIO,   // 우선순위 (기본: 5)
        nullptr);                         // task handle 저장 안 함
}
```

---

## 41. CMakeLists.txt 작성
```bash
nano main/CMakeLists.txt
```

#### 작성:
```txt
idf_component_register(
    SRCS
        "main.cpp"

    INCLUDE_DIRS
        "."
)
```

---

## 42. WIFI 설정
```bash
idfpy
cd ~/robot_navigation/firmware/esp32

idf.py clean-microros
idf.py reconfigure

idf.py menuconfig
```

<br>

#### 설정:
```text
"micro-ROS Settings" -> "micro-ROS network interface select" -> "WLAN interface" -> "WiFi Configuration" -> "WiFi SSID" -> "WiFi Password" -> Esc -> "micro-ROS Agent IP" -> S로 저장 -> Esc
```

<br>

#### 확인:
```text
(venv) grep CONFIG_MICRO_ROS_ESP_NETIF_WLAN sdkconfig

>> CONFIG_MICRO_ROS_ESP_NETIF_WLAN=y
(venv) grep CONFIG_MICRO_ROS_AGENT sdkconfig
>> CONFIG_MICRO_ROS_AGENT_IP="xxx.xx..."
>> CONFIG_MICRO_ROS_AGENT_PORT="8888"

(venv) grep CONFIG_ESP_WIFI_SSID sdkconfig
>> CONFIG_ESP_WIFI_SSID="my_wifi_name"
```

---

## 43. 빌드, Flash, 모니터
```bash
idf.py build
idf.py -p /dev/ttyUSB0 flash monitor
```

```text
>> I (xxxxx) wifi_station_netif: connected to ap SSID:xxxx password:xxxx
```

---

## 44. Ubuntu firewall 확인
```bash
sudo ufw status
```

```text
>> Status: inactive
이 경우 다음 단계 스킵
```

```text
>> Status: active
이 경우 다음 단계 진행
```

<br>

```bash
sudo ufw allow 8888/udp
sudo ufw status
```

```text
>> Status: inactive
```

---

## 45. micro-ROS Agent 실행 및 ROS2 토픽 통신 테스트


#### 새 터미널에서:

```bash
echo $ROS_DOMAIN_ID
```

```text
>> 본인 ROS_DOMAIN_ID
```

<br>

```bash
ros2 run micro_ros_agent micro_ros_agent \
    udp4 \
    --port 8888 \
    -v6
```

- 리셋 버튼

<br>

```bash
ros2 topic pub -r 2 /esp32/cmd std_msgs/msg/Int32 "{data: 123456789}"
```

---

## 46. 로봇 주행 (키보드 Teleop 모터 제어)


#### 패키지 설치 확인:

```bash
ros2 pkg executables teleop_twist_keyboard
```

```text
>> teleop_twist_keyboard teleop_twist_keyboard
```

#### 패키지 없을 시:

```bash
sudo apt update
sudo apt install ros-jazzy-teleop-twist-keyboard
```

<br>

#### 기존 파일 백업

```bash
cp main/main.cpp main/main.cpp.lidar_check.backup
cp main/CMakeLists.txt main/CMakeLists.txt.lidar_check.backup
```

<br>

- `main.cpp` 작성
- `CMakeLists.txt` 작성

<br>

#### 터미널 \#1 (idfsh)

```bash
idfsh
cd ~/robot_navigation/firmware/esp32

idf.py build
idf.py -p /dev/ttyUSB0 flash monitor
```

#### 터미널 \#2 (Agent)

```bash
ros2 run micro_ros_agent micro_ros_agent \
    udp4 \
    --port 8888 \
    -v6
```

#### 터미널 \#3 (teleop)

```bash
ros2 run teleop_twist_keyboard teleop_twist_keyboard
```

#### 터미널 \#4 (별도 창)

```bash
ros2 topic list
ros2 topic info /cmd_vel -v
ros2 topic echo /cmd_vel
```

---

## 47. turtlesim 실행 (QoS 확인)

```bash
ros2 run turtlesim turtlesim_node
ros2 run turtlesim turtle_teleop_key
ros2 node list
ros2 topic list
rqt_graph
ros2 topic echo /turtle1/cmd_vel
ros2 topic info /turtle1/cmd_vel
ros2 topic info /turtle1/cmd_vel -v
```

---

## 48. 시뮬레이션 환경 설정 (TurtleBot3 Burger)

```bash
echo "export TURTLEBOT3_MODEL=burger" >> ~/.bashrc
source ~/.bashrc
```

---

## 49. Gazebo 시뮬레이션 첫 시작

```bash
ros2 launch turtlebot3_gazebo turtlebot3_dqn_stage1.launch.py
```

- topic을 publish하여 시뮬레이션 로봇 주행

---

## 50. RViz 실행 및 LaserScan 확인

```bash
rviz2
```

- Fixed Frame 설정
- Add - By display type: RobotModel
- Add - By display type: LaserScan
    - Topic 설정: /scan
    - QoS 설정: Reliability - Best Effort
---

## 51. Odometry 확인

```bash
ros2 topic echo /odom
```

---

## 52. TF 확인 (토픽 및 그래프)

```bash
ros2 topic echo /tf
ros2 run tf2_tools view_frames
```

---
---

## A. Git

**git 최초 실행:**

```bash
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
```
<br>

**로그인 정보 저장:**
```bash
git config --global credential.helper store
```

---

## B. VS Code 설치

```bash
wget -O vscode.deb "https://update.code.visualstudio.com/latest/linux-deb-arm64/stable"
sudo apt install ./vscode.deb
```

---

## C. Linux CLI

#### $1)$ 터미널 프롬프트 (prompt) 이해

터미널을 실행하면 보이는 문자열:

```bash
user_name@host_name:~/parent_directory/current_directory$
```

* user_name: 현재 로그인한 사용자 이름
* host_name: 현재 접속한 컴퓨터(호스트) 이름
* @: 사용자와 컴퓨터를 구분하는 표시
* ~: 사용자의 home 디렉토리(폴더)
* /: 폴더와 폴더를 구분하는 표시
* parent_directory: 상위(부모) 디렉토리
* current_directory: 현재 디렉토리
* $: 일반 사용자 권한을 나타내는 표시 (#: 관리자 권한)

#### $2)$ 기본 터미널 명령어 종류

(a) `cd`: 디렉토리 이동 (change directory)

```bash
cd target_directory
```

<br>(b) `cd`: 홈 디렉토리로 이동

```bash
cd
```

<br>(c) `cd ..`: 상위 디렉토리로 이동

```bash
cd ..
```

<br>(d) `pwd`: 현재 디렉토리 확인 (print working directory)

```bash
pwd
```

<br>(e) `ls`: 현재 디렉토리 내 파일 목록 확인 (list)

```bash
ls
```

<br>(f) `ls -a`: 현재 디렉토리 내 숨김 파일 포함 모든 파일 확인 (list -all)

```bash
ls -a
```

<br>(g) `mkdir`: 현재 경로에 새로운 폴더 생성 (make directory)

```bash
mkdir folder_name
```

<br>(h) `mkdir -p`: 현재 경로에 새로운 폴더 생성하며 상위 폴더까지 생성 (make directory -parent)

```bash
mkdir -p parent_folder_name/folder_name
```

<br>(i) `touch`: 현재 경로에 새로운 파일 생성

```bash
touch file_name.txt
```

<br>(j) `cp`: 파일 복사 (copy)

```bash
cp file_path/file_name copy_path/new_file_name
```

<br>(k) `mv`: 파일 이동 (move)

```bash
mv file_path/file_name move_path/new_file_name
```

<br>(l) `rm`: 파일 삭제 (remove)

```bash
rm file_path/file_name
```

<br>(m) `rm -r`: 해당 폴더 및 모든 내부 파일 삭제 (remove -recursive)

```bash
rm -r folder_path
```

<br>(n) rmdir: 빈 폴더 삭제 (remove directory)

```bash
rmdir empty_folder
```

<br>(o) echo: 입력한 문자열이나 값을 터미널에 출력 혹은 파일에 저장

```bash
echo hi
echo "hi"
echo $USER
echo "hi" > test.txt
echo "hello" > test.txt
echo "again" >> test.txt
```

<br>(p) chmod: 파일/폴더의 권한 변경 (change mode)

```bash
chmod +x file_name
chmod -x file_name
chmod +w file_name
chmod -w file_name
chmod 444 file_name
chmod 644 file_name
```

<br>(q) nano: 터미널에서 GUI없이 사용하는 텍스트 편집기

```bash
nano file_name
```

<br>(r) ping: 특정 컴퓨터(호스트)와 네트워크 연결 상태를 확인

```bash
ping 8.8.8.8
ping <ip_address>
ping <host_name>.local
```

IP 주소 확인법:

```bash
hostname -I
```

<br>(s) ssh: 원격 컴퓨터(Jetson)에 접속

```bash
ssh <user_name>@<host_name>
ssh <user_name>@<ip_address>
```

<br>(t) sudo: 관리자(root) 권한으로 명령어 실행

```bash
sudo 명령어
sudo rm 관리자_권한_폴더/관리자_권한_파일
sudo nano 관리자_권한_폴더/관리자_권한_파일
```

가장 많이 사용되는 `sudo` 명령어:<br>
`sudo apt update`: 패키지 목록 갱신<br>
`sudo apt upgrade`: 설치된 패키지를 최신 버전으로 업데이트<br>
`sudo apt install <package_name>`: 패키지 설치

---

## D. PCB-ESP32 GPIO 구성

```cpp
// Motor
#define FR_IN1 26
#define FR_IN2 25

#define FL_IN1 19
#define FL_IN2 27

#define RR_IN1 23
#define RR_IN2 12

#define RL_IN1 0
#define RL_IN2 15

// IMU (MPU6050)
#define IMU_SDA 21
#define IMU_SCL 22

// LiDAR
#define LIDAR_RX 16

// Potentiometer
#define POT_PIN 39

// Servo
#define SERVO_PIN 14

// Buzzer
#define BUZZER_PIN 5
```

---

## E. 유용한 Commands:

```bash
ros2 run [pkg] [executable]
ros2 run [pkg] [executable] --ros-args -p [param]:=[value]

ros2 node list
ros2 node info /node

ros2 topic list
ros2 topic list -t
ros2 topic list -v
ros2 topic echo /topic
ros2 topic info /topic
ros2 topic info /topic -v
ros2 topic pub /topic [msg_type] "[data]"
ros2 topic hz
ros2 topic find [msg_type]

ros2 service list
ros2 service list -t
ros2 service type /service
ros2 service find [srv_type]
ros2 service call /service [srv_type] "[data]"

ros2 action list
ros2 action list -t
ros2 action info /action
ros2 action info /action -t
ros2 action send_goal /action [action_type] "[data]"
ros2 action send_goal /action [action_type] "[data]" --feedback

ros2 param list
ros2 param get /node [param]
ros2 param set /node [param] [value]
ros2 param describe /node [param]

ros2 interface list
ros2 interface show [interface]
ros2 interface packages

ros2 doctor
ros2 doctor --report
ros2 wtf

rqt_graph
ros2 run tf2_tools view_frames
rviz2 -d [PATH]/[CONFIG_FILE]
```

---